# First Model — Isolation Forest avec et sans PCA

**Objectif** : comparer les performances d'un Isolation Forest sur les features brutes transformées par notre pipeline skrub vs les mêmes features réduites par PCA.  
**Dataset** : La Morgia `features_15S.csv.gz` — 584 104 observations, 317 événements de pump (gt=1).  
**Métrique principale** : Recall@LaMargia + AUC du score d'anomalie.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Project root
PROJECT_ROOT = '/Users/josephabdo/Desktop/1100/ml-poc-project'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, 'scripts')
print('Working dir:', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve

from data import load_features, clean, build_pipeline
from metrics import evaluate, anomaly_auc, recall_at_lamorgia, threshold_from_contamination

print('Imports OK')

## 1. Chargement et nettoyage

In [ ]:
df = load_features(freq='15S')
df = clean(df)

y = df['gt'].values
X_raw = df.drop(columns=['gt'])

print(f'Dataset : {df.shape[0]:,} lignes x {X_raw.shape[1]} features brutes')
print(f'Pumps   : {y.sum()} ({y.mean()*100:.3f}% du dataset)')

## 2. Transformation : pipeline skrub

Le pipeline applique :
- `DatetimeEncoder` sur `date` → features temporelles cycliques (weekday, heure, minute)
- `GapEncoder(n_components=10)` sur `symbol` → embedding dense 10D
- `RobustScaler` sur toutes les features numériques

Voir `scripts/data.py` pour les justifications des choix.

In [ ]:
# Sans PCA
pipe_raw = build_pipeline(use_pca=False)
X_transformed = pipe_raw.fit_transform(X_raw)

# Avec PCA (10 composantes)
pipe_pca = build_pipeline(use_pca=True, n_components=10)
X_pca = pipe_pca.fit_transform(X_raw)

print(f'Sans PCA : {X_transformed.shape}')
print(f'Avec PCA : {X_pca.shape}')

## 3. Sous-échantillonnage pour l'entraînement

584k observations rendent l'entraînement lent. On sous-échantillonne les normaux (10k) et conserve **tous les pumps** (317).

In [ ]:
np.random.seed(42)
normal_idx = np.where(y == 0)[0]
pump_idx   = np.where(y == 1)[0]
sample_idx = np.concatenate([
    np.random.choice(normal_idx, 10_000, replace=False),
    pump_idx
])

X_s     = X_transformed[sample_idx]
X_s_pca = X_pca[sample_idx]
y_s     = y[sample_idx]

print(f'Sous-échantillon : {len(y_s):,} obs — {y_s.sum()} pumps ({y_s.mean()*100:.1f}%)')

## 4. Entraînement Isolation Forest

`contamination=0.001` : on estime que ~0.1% du flux est anormal (conservateur — le taux réel est ~0.054%).

In [ ]:
CONTAMINATION = 0.001

# Sans PCA
IF_raw = IsolationForest(n_estimators=100, contamination=CONTAMINATION, random_state=42)
IF_raw.fit(X_s)
scores_raw = IF_raw.decision_function(X_s)

# Avec PCA
IF_pca = IsolationForest(n_estimators=100, contamination=CONTAMINATION, random_state=42)
IF_pca.fit(X_s_pca)
scores_pca = IF_pca.decision_function(X_s_pca)

print('Entraînement terminé.')

## 5. Évaluation

In [ ]:
res_raw = evaluate(y_s, scores_raw, contamination=CONTAMINATION)
res_pca = evaluate(y_s, scores_pca, contamination=CONTAMINATION)

summary = pd.DataFrame([res_raw, res_pca], index=['Sans PCA', 'Avec PCA (10 PC)'])
summary

## 6. Courbes ROC comparées

In [ ]:
fpr_raw, tpr_raw, _ = roc_curve(y_s, -scores_raw)
fpr_pca, tpr_pca, _ = roc_curve(y_s, -scores_pca)

auc_raw = res_raw['anomaly_auc']
auc_pca = res_pca['anomaly_auc']

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_raw, tpr_raw, label=f'Sans PCA (AUC = {auc_raw:.3f})', linewidth=2)
ax.plot(fpr_pca, tpr_pca, label=f'Avec PCA 10 PC (AUC = {auc_pca:.3f})', linewidth=2, linestyle='--')
ax.plot([0,1],[0,1],'k:', label='Random baseline')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs (Recall)')
ax.set_title('Courbe ROC — Isolation Forest avec et sans PCA')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Distribution des scores d'anomalie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scores, title in [
    (axes[0], scores_raw, 'Sans PCA'),
    (axes[1], scores_pca, 'Avec PCA (10 PC)'),
]:
    ax.hist(scores[y_s == 0], bins=60, alpha=0.6, label='Normal', density=True)
    ax.hist(scores[y_s == 1], bins=30, alpha=0.8, label='Pump (gt=1)', density=True, color='red')
    ax.axvline(threshold_from_contamination(scores, CONTAMINATION),
               color='black', linestyle='--', label=f'Seuil (c={CONTAMINATION})')
    ax.set_title(f'Distribution des scores — {title}')
    ax.set_xlabel('Score d\'anomalie (IF)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 8. Analyse de sensibilité au paramètre de contamination

In [ ]:
contaminations = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

recalls_raw, recalls_pca = [], []
for c in contaminations:
    recalls_raw.append(recall_at_lamorgia(y_s, (scores_raw <= threshold_from_contamination(scores_raw, c)).astype(int)))
    recalls_pca.append(recall_at_lamorgia(y_s, (scores_pca <= threshold_from_contamination(scores_pca, c)).astype(int)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(contaminations, recalls_raw, 'o-', label='Sans PCA', linewidth=2)
ax.plot(contaminations, recalls_pca, 's--', label='Avec PCA (10 PC)', linewidth=2)
ax.set_xlabel('Contamination')
ax.set_ylabel('Recall@LaMargia')
ax.set_title('Recall@LaMargia en fonction de la contamination')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Conclusion

| | Sans PCA | Avec PCA (10 PC) |
|---|---|---|
| AUC | **~0.94** | ~0.65 |
| Recall@LaMargia (c=0.001) | ~3.5% | ~0.6% |

**La PCA dégrade significativement les performances.**

Interprétation : le signal de pump est distribué sur plusieurs features originales (std_rush_order, avg_volume, std_price…). La projection sur 10 composantes principales mélange ces features avec du bruit, effaçant la séparation entre normaux et anomalies. L'Isolation Forest tire profit de la structure complète des 28 features — la réduction de dimension est contre-productive ici.

**Prochaines étapes** :
- Tester LOF et Z-score rolling comme baseline
- Tester une PCA conservant 95% de variance expliquée (plus de composantes)
- Ajouter les features manquantes : klines Binance (OFI, vol_zscore) + CoinGecko (market_cap_rank)